# Notebook 01: Raw Data Audit & Integrity Profiling
**Project:** E-Commerce Business Intelligence & Customer Analytics Platform  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (99,441 records)

This notebook performs an end-to-end data audit across all 9 raw Olist CSV datasets, profiling row counts, schemas, missing values, primary key uniqueness, and temporal bounds.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingestion.load_raw_data import load_all_raw_data, audit_raw_data
from src.preprocessing.clean_data import DataCleaner
from src.validation.reconciliation import run_reconciliation

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Load All 9 Raw Tables

In [ ]:
raw_tables = load_all_raw_data()
audit_summary = audit_raw_data(raw_tables)

print(f"Total Tables Loaded: {audit_summary['_summary']['total_tables']}")
print(f"Total Raw Records:   {audit_summary['_summary']['total_rows']:,}")
print(f"Total Memory Usage:  {audit_summary['_summary']['total_memory_mb']} MB")

## 2. Table Summary & Null Value Profiling

In [ ]:
profile_rows = []
for name, info in audit_summary.items():
    if name != '_summary':
        profile_rows.append({
            'Table': name,
            'Rows': info['row_count'],
            'Columns': info['column_count'],
            'Memory (MB)': info['memory_mb'],
            'Columns with Nulls': len(info['columns_with_nulls'])
        })

pd.DataFrame(profile_rows)

## 3. Order Status Breakdown

In [ ]:
orders_df = raw_tables['orders']
status_counts = orders_df['order_status'].value_counts()
print("Order Status Frequencies:")
print(status_counts)

plt.figure(figsize=(9, 4))
status_counts.plot(kind='bar', color='#1E3A8A')
plt.title('Raw Order Status Distribution')
plt.ylabel('Order Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Run Cleaning Pipeline and Reconcile

In [ ]:
cleaner = DataCleaner()
cleaned_tables, cleaning_audit = cleaner.clean_all(raw_tables)
recon_report = run_reconciliation(raw_tables, cleaned_tables)
recon_report[['metric', 'source_raw', 'target_transformed', 'variance', 'pct_reconciled']]